In [56]:
import subprocess
import glob
import re

import pandas as pd
import matplotlib.pyplot as plt

In [57]:
files = glob.glob("results_p*.txt")

In [60]:
output = subprocess.check_output(["grep", "score information", *files],text=True)
lines = output.strip().splitlines()

# Compile regex once
pattern = re.compile(
    r"Macro=(?P<Macro>[0-9.]+)\s+Weighted=(?P<Weighted>[0-9.]+)\s+Micro=(?P<Micro>[0-9.]+)"
)

records = []
for line in lines:
    # Example: split into filename + content
    if ":" in line:  # grep prefixes filename when searching multiple files
        filename, content = line.split(":", 1)
        content = content.strip()
        _, pods, ss, spines, leafs, hosts, pps = filename.split('_')
        
        match = pattern.search(content)
 
        records.append({'filename': filename,
                        'pods': int(pods[-1]),
                        'super_spines': int(ss[-1]),
                        'spines': int(spines[-1]),
                        'leafs': int(leafs[-1]),
                        'hosts_per_leaf': int(hosts[-1]),
                        'pps': int(pps.strip('pps.txt')),
                        'Macro': float(match.group("Macro")),
                        'Weighted': float(match.group("Weighted")),
                        'Micro': float(match.group("Micro")),
                       })
scores = pd.DataFrame(records).set_index('filename')

In [61]:
output = subprocess.check_output(["grep", "Collision count:", *files],text=True)
lines = output.strip().splitlines()


records = []
for line in lines:
    # Example: split into filename + content
    if ":" in line:  # grep prefixes filename when searching multiple files
        filename, content = line.split(":", 1)
        content = content.strip()
 
        records.append({'filename': str(filename),
                        'collisions': int(content.split(':')[-1].strip())
                       })
collisions = pd.DataFrame(records).set_index('filename')

In [62]:
output = subprocess.check_output(["grep", "                 'support':", *files],text=True)
lines = output.strip().splitlines()


records = []
for line in lines:
    # Example: split into filename + content
    if ":" in line:  # grep prefixes filename when searching multiple files
        filename, content = line.split(":", 1)
        content = content.strip()
 
        records.append({'filename': filename,
                        'support': float(content.split(':')[-1].strip()[:-2])
                       })
support = pd.DataFrame(records).set_index('filename')

In [63]:
results = scores.join(collisions).join(support)

In [64]:
results.sort_values(by='Macro', ascending=False)

,pods,super_spines,spines,leafs,hosts_per_leaf,pps,Macro,Weighted,Micro,collisions,support
filename,,,,,,,,,,,
results_p2_ss1_s3_l3_h1_20pps.txt,2,1,3,3,1,20,0.665809,0.709455,0.716358,351821,105278.0
results_p1_ss1_s2_l2_h1_100pps.txt,1,1,2,2,1,100,0.662866,0.700886,0.706629,158497,60169.0
results_p1_ss1_s3_l3_h1_100pps.txt,1,1,3,3,1,100,0.659103,0.699546,0.705234,180759,60169.0
results_p1_ss1_s4_l4_h2_100pps.txt,1,1,4,4,2,100,0.658445,0.700450,0.706396,272987,60169.0
results_p1_ss1_s2_l2_h1_200pps.txt,1,1,2,2,1,200,0.656666,0.693262,0.699715,172693,60169.0
results_p3_ss1_s3_l3_h1_100pps.txt,3,1,3,3,1,100,0.655817,0.696451,0.702477,235790,60169.0
results_p2_ss1_s3_l3_h1_100pps.txt,2,1,3,3,1,100,0.655569,0.695130,0.701465,233747,60169.0
results_p2_ss1_s4_l4_h2_100pps.txt,2,1,4,4,2,100,0.654359,0.697597,0.703672,214635,60169.0
results_p1_ss1_s3_l4_h1_100pps.txt,1,1,3,4,1,100,0.654149,0.691000,0.697892,305703,60169.0


In [ ]:
results['hosts'] = results.pods*results.hosts_per_leaf*results.leafs

In [ ]:
results.plot(kind = 'scatter', x = 'hosts', y = 'collisions')